# ALS Forest Height vs Biomass product Forest Height

The script below queries the ALS (Global Canopy Atlas) and the Biomass L2A Forest Height products from the ESA MAAP catalog, cmpare both datasets and more analyses.

### 1. Import packages

In [ ]:
from shapely.geometry import Polygon
from rasterio.windows import Window
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
from pystac_client import Client
from folium import GeoJson
import geopandas as gpd
from io import BytesIO
import rasterio as rio
from tqdm import tqdm
from PIL import Image
import pandas as pd
import numpy as np
import requests
import pathlib
import fsspec
import folium
import os

from shapely.geometry import shape
from rasterio.enums import Resampling
import rioxarray 


### 2. Query the ESA MAAP STAC catalog

In [ ]:
catalog_url = 'https://catalog.maap.eo.esa.int/catalogue/'
catalog = Client.open(catalog_url)

In [ ]:
results = catalog.collection_search(
    filter="platform='Air-based Platform'",
    datetime="2025-06-01T00:00:00.000Z/..",
)
print(f"{results.matched()} collections found.")

In [ ]:
coll = catalog.get_collection("BiomassLevel2a")
print(coll.title)

search = catalog.search(
    collections=["BiomassLevel2a"],
    filter="productType='FP_FH__L2A'",
    filter_lang="cql2-text",
    datetime="2025-06-01T00:00:00Z/..",
    # bbox=[lon_min, lat_min, lon_max, lat_max], 
    max_items=50,
)

items = list(search.items())
print(f"{len(items)} forest height items found")

df = pd.DataFrame(
    [{"id": i.id, "datetime": i.datetime, **i.properties} for i in items]
)
df[["id", "datetime"]].head()

### 3. ALS collections and footprints

In [ ]:
# Show the ALS collections
als_df = pd.json_normalize(results.collection_list_as_dict(), record_path=["collections"])
ALS_COLLECTIONS = als_df["id"].tolist() 
als_df[["id", "title", "extent.temporal.interval", "extent.spatial.bbox"]]

In [ ]:
def items_to_gdf(items, source):
    rows = [{
        "id": it.id,
        "datetime": pd.Timestamp(it.datetime),
        "source": source,
        "geometry": shape(it.geometry),
        "item": it,
    } for it in items]
    return gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326")

# Count ALS items
als_search = catalog.search(collections=ALS_COLLECTIONS, limit=100, max_items=500)
print("ALS items matched:", als_search.matched())

als_items = []
for it in tqdm(als_search.items(), desc="ALS"):
    als_items.append(it)
gdf_als = items_to_gdf(als_items, "ALS")
print("ALS bbox:", gdf_als.total_bounds.round(2))

In [ ]:
# Look for forest height L2A overlapping with the ALS area
bbox = list(gdf_als.total_bounds)
fh_search = catalog.search(
    collections=["BiomassLevel2a"],
    filter="productType='FP_FH__L2A'",
    filter_lang="cql2-text",
    bbox=bbox,
    limit=100,
    max_items=500,
)
print("FH L2A items matched:", fh_search.matched())

fh_items = []
for it in tqdm(fh_search.items(), desc="FH L2A"):
    fh_items.append(it)
gdf_fh = items_to_gdf(fh_items, "BIOMASS FH L2A")
print(f"ALS items: {len(gdf_als)}   |   FH L2A items: {len(gdf_fh)}")

### 4. Spatial and temporal overlap

In [ ]:
from shapely.validation import make_valid

# Repair invalid footprints
for g in (gdf_als, gdf_fh):
    print(f"invalid geometries: {(~g.is_valid).sum()} / {len(g)}")
    g["geometry"] = g.geometry.apply(make_valid)

pairs = gpd.sjoin(gdf_als, gdf_fh, how="inner", predicate="intersects",
                  lsuffix="als", rsuffix="fh")

# overlap area in km² (equal-area projection)
ea = "EPSG:6933"
als_geom = gdf_als.drop_duplicates("id").set_index("id").to_crs(ea).geometry
fh_geom = gdf_fh.drop_duplicates("id").set_index("id").to_crs(ea).geometry


def overlap_km2(a, f):
    try:
        return als_geom[a].intersection(fh_geom[f]).area / 1e6
    except Exception:
        # last resort for anything make_valid didn't fix
        return als_geom[a].buffer(0).intersection(fh_geom[f].buffer(0)).area / 1e6


pairs["overlap_km2"] = [overlap_km2(a, f) for a, f in zip(pairs["id_als"], pairs["id_fh"])]
pairs = pairs[pairs["overlap_km2"] > 0]          # drop pairs that only touch at an edge
pairs["dt_days"] = (pairs["datetime_fh"] - pairs["datetime_als"]).dt.days

print(f"Overlapping ALS/FH pairs: {len(pairs)}")
print(f"ALS tiles with >=1 BIOMASS overlap: {pairs['id_als'].nunique()} / {len(gdf_als)}")
print(f"Total overlap area: {pairs['overlap_km2'].sum():.1f} km²")
pairs[["id_als", "id_fh", "overlap_km2", "dt_days"]].sort_values("overlap_km2", ascending=False).head(10)

In [ ]:
# Time gap between ALS flight and BIOMASS acquisition
print(pairs["dt_days"].describe().round(0))
print("ALS acquisition years:")
print(gdf_als["datetime"].dt.year.value_counts().sort_index())

covered = gdf_als[gdf_als["id"].isin(pairs["id_als"])]
print(f"ALS area with BIOMASS coverage: {covered.to_crs('EPSG:6933').area.sum() / 1e6:.0f} km²")

# number of BIOMASS products overlapping each ALS tile
print(pairs.groupby("id_als").size().value_counts().sort_index())

In [ ]:
from matplotlib.patches import Patch

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
gdf_fh.boundary.plot(ax=ax[0], color="tab:orange", lw=0.6)
gdf_als.plot(ax=ax[0], color="tab:green", alpha=0.6)
ax[0].set_title("Footprints")
ax[0].legend(handles=[Patch(edgecolor="tab:orange", facecolor="none", label="BIOMASS FH L2A"),
                      Patch(color="tab:green", alpha=0.6, label="ALS")])

ax[1].hist(pairs["dt_days"], bins=30, color="grey")
ax[1].set_xlabel("BIOMASS date − ALS date (days)")
ax[1].set_ylabel("# overlapping pairs")
ax[1].set_title("Temporal gap between ALS and BIOMASS")
plt.tight_layout()
plt.show()

In [ ]:
# Interactive map 
m = folium.Map(location=[np.mean(bbox[1::2]), np.mean(bbox[0::2])], zoom_start=8)
GeoJson(gdf_fh[["id", "geometry"]], name="BIOMASS FH L2A",
        style_function=lambda f: {"color": "orange", "weight": 1, "fillOpacity": 0}).add_to(m)
GeoJson(gdf_als[["id", "geometry"]], name="ALS",
        style_function=lambda f: {"color": "green", "weight": 1, "fillOpacity": 0.4}).add_to(m)
folium.LayerControl().add_to(m)
m

### 5. Open one overlapping pair
Picks the ALS/BIOMASS pair with the largest overlap among pairs with maximum 2 years difference

In [ ]:
# Keep pairs at most 2 years difference
MAX_GAP_DAYS = 730
recent = pairs[pairs["dt_days"].abs() <= MAX_GAP_DAYS]
best_pairs = (recent.sort_values("dt_days", key=abs)
                    .drop_duplicates("id_als")
                    .sort_values("overlap_km2", ascending=False))
print(f"Pairs within {MAX_GAP_DAYS} days: {len(recent)} | ALS tiles kept: {len(best_pairs)}")

# select the best ALS/Biomass pair (=top row)
best = best_pairs.iloc[0]
als_item = gdf_als.set_index("id").loc[best["id_als"], "item"]
fh_item = gdf_fh.set_index("id").loc[best["id_fh"], "item"]
print(f"Using pair: {als_item.id}  +  {fh_item.id}  ({best['dt_days']} days apart)")

for name, it in [("ALS", als_item), ("FH", fh_item)]:
    print(f"\n--- {name}: {it.id}")
    for k, a in it.assets.items():
        print(f"  {k:30s} {a.media_type}")

In [ ]:
ALS_ASSET = "enclosure_chm_lspikefree_tif"   # ALS canopy height model (spike free)
FH_ASSET = "enclosure_i_fh_tiff"             # BIOMASS FH
FHQ_ASSET = "enclosure_i_quality_tiff"       # BIOMASS FH quality

TOKEN = os.environ["MAAP_CATALOGUE_ACCESS_TOKEN"]

r = requests.head(als_item.assets[ALS_ASSET].href,
                  headers={"Authorization": f"Bearer {TOKEN}"}, allow_redirects=True)
#print("ALS with environment token:", r.status_code)

with rio.Env(GDAL_HTTP_HEADERS=f"Authorization: Bearer {TOKEN}"):
    als = rioxarray.open_rasterio(als_item.assets[ALS_ASSET].href, masked=True).squeeze().load()

    # cut the BIOMASS frame to the ALS tile
    als_bounds = als.rio.transform_bounds("EPSG:4326")
    fh = (rioxarray.open_rasterio(fh_item.assets[FH_ASSET].href, masked=True).squeeze()
          .rio.clip_box(*als_bounds, crs="EPSG:4326").load())
    fhq = (rioxarray.open_rasterio(fh_item.assets[FHQ_ASSET].href, masked=True).squeeze()
           .rio.clip_box(*als_bounds, crs="EPSG:4326").load())

for name, da in [("ALS", als), ("FH L2A", fh), ("FH qual", fhq)]:
    print(f"{name:8s} CRS={da.rio.crs}  res={da.rio.resolution()}  shape={da.shape}  "
          f"range=({float(da.min()):.1f}, {float(da.max()):.1f})")

### 6. Put ALS on the BIOMASS grid
ALS (~1 m2 resolution) is aggregated inside each BIOMASS pixel: with mean height and max height. Keeping only pixels with ≥80% ALS coverage

In [ ]:
als_on_fh_mean = als.rio.reproject_match(fh, resampling=Resampling.average)
als_on_fh_max = als.rio.reproject_match(fh, resampling=Resampling.max)
valid = als.notnull().astype("float32").rio.write_nodata(np.nan)
coverage = valid.rio.reproject_match(fh, resampling=Resampling.average)

MIN_COVER = 0.8
cmp = pd.DataFrame({
    "fh": fh.values.ravel(),
    "als_mean": als_on_fh_mean.values.ravel(),
    "als_max": als_on_fh_max.values.ravel(),
    "cover": coverage.values.ravel(),
}).dropna()
cmp = cmp[cmp["cover"] >= MIN_COVER]
print(f"Matched BIOMASS pixels with >={MIN_COVER:.0%} ALS coverage: {len(cmp)}")


def stats(y_ref, y_est):
    d = y_est - y_ref
    r = np.corrcoef(y_ref, y_est)[0, 1]
    return {"n": len(d), "bias": d.mean(), "RMSE": np.sqrt((d**2).mean()),
            "rel_RMSE_%": 100 * np.sqrt((d**2).mean()) / y_ref.mean(), "R2": r**2}


pd.DataFrame({k: stats(cmp[k], cmp["fh"]) for k in ["als_mean", "als_max"]}).T.round(2)

### 7. First plots comparing ALS and Biomass FH

In [ ]:
# with max height
fig, ax = plt.subplots(1, 3, figsize=(16, 5))
vmax = float(np.nanpercentile(fh, 99))
fh.plot(ax=ax[0], vmin=0, vmax=vmax, cmap="viridis"); ax[0].set_title("BIOMASS FH L2A")
als_on_fh_max.plot(ax=ax[1], vmin=0, vmax=vmax, cmap="viridis"); ax[1].set_title("ALS (max, on BIOMASS grid)")
(fh - als_on_fh_max).plot(ax=ax[2], cmap="RdBu_r", center=0); ax[2].set_title("BIOMASS − ALS")
for a in ax:
    a.set_aspect("equal")
plt.tight_layout()
plt.show()

# with mean height
fig, ax = plt.subplots(1, 3, figsize=(16, 5))
vmax = float(np.nanpercentile(fh, 99))
fh.plot(ax=ax[0], vmin=0, vmax=vmax, cmap="viridis"); ax[0].set_title("BIOMASS FH L2A")
als_on_fh_mean.plot(ax=ax[1], vmin=0, vmax=vmax, cmap="viridis"); ax[1].set_title("ALS (mean, on BIOMASS grid)")
(fh - als_on_fh_mean).plot(ax=ax[2], cmap="RdBu_r", center=0); ax[2].set_title("BIOMASS − ALS")
for a in ax:
    a.set_aspect("equal")
plt.tight_layout()
plt.show()

In [ ]:
ref = "als_mean"   # Or "als_max" to compare
lim = max(cmp[ref].max(), cmp["fh"].max())

fig, ax = plt.subplots(1, 3, figsize=(17, 5))
hb = ax[0].hexbin(cmp[ref], cmp["fh"], gridsize=50, mincnt=1, cmap="magma_r")
ax[0].plot([0, lim], [0, lim], "k--", lw=1, label="1:1")
ax[0].set_xlabel(f"ALS height ({ref}) [m]"); ax[0].set_ylabel("BIOMASS FH [m]")
s = stats(cmp[ref], cmp["fh"])
ax[0].set_title(f"n={s['n']}  bias={s['bias']:.1f} m  RMSE={s['RMSE']:.1f} m  R²={s['R2']:.2f}")
ax[0].legend(); plt.colorbar(hb, ax=ax[0], label="count")

bins = np.linspace(0, lim, 40)
ax[1].hist(cmp[ref], bins=bins, alpha=0.6, label="ALS")
ax[1].hist(cmp["fh"], bins=bins, alpha=0.6, label="BIOMASS")
ax[1].set_xlabel("Height [m]"); ax[1].set_title("Height distributions"); ax[1].legend()

# residuals per ALS height class -> saturation in tall forest / overestimation in low forest?
cmp["h_class"] = pd.cut(cmp[ref], bins=np.arange(0, lim + 5, 5))
cmp["resid"] = cmp["fh"] - cmp[ref]
cmp.boxplot(column="resid", by="h_class", ax=ax[2], rot=45, showfliers=False)
ax[2].axhline(0, color="k", lw=1)
ax[2].set_xlabel("ALS height class [m]"); ax[2].set_ylabel("BIOMASS − ALS [m]")
ax[2].set_title("Residuals by height class"); fig.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
ref = "als_max"   # Or "als_mean" to compare
lim = max(cmp[ref].max(), cmp["fh"].max())

fig, ax = plt.subplots(1, 3, figsize=(17, 5))
hb = ax[0].hexbin(cmp[ref], cmp["fh"], gridsize=50, mincnt=1, cmap="magma_r")
ax[0].plot([0, lim], [0, lim], "k--", lw=1, label="1:1")
ax[0].set_xlabel(f"ALS height ({ref}) [m]"); ax[0].set_ylabel("BIOMASS FH [m]")
s = stats(cmp[ref], cmp["fh"])
ax[0].set_title(f"n={s['n']}  bias={s['bias']:.1f} m  RMSE={s['RMSE']:.1f} m  R²={s['R2']:.2f}")
ax[0].legend(); plt.colorbar(hb, ax=ax[0], label="count")

bins = np.linspace(0, lim, 40)
ax[1].hist(cmp[ref], bins=bins, alpha=0.6, label="ALS")
ax[1].hist(cmp["fh"], bins=bins, alpha=0.6, label="BIOMASS")
ax[1].set_xlabel("Height [m]"); ax[1].set_title("Height distributions"); ax[1].legend()

# residuals per ALS height class -> saturation in tall forest / overestimation in low forest?
cmp["h_class"] = pd.cut(cmp[ref], bins=np.arange(0, lim + 5, 5))
cmp["resid"] = cmp["fh"] - cmp[ref]
cmp.boxplot(column="resid", by="h_class", ax=ax[2], rot=45, showfliers=False)
ax[2].axhline(0, color="k", lw=1)
ax[2].set_xlabel("ALS height class [m]"); ax[2].set_ylabel("BIOMASS − ALS [m]")
ax[2].set_title("Residuals by height class"); fig.suptitle("")
plt.tight_layout()
plt.show()

### 8. Run the comparison on multiple ALS tiles
Wrapped in a function and looped over `best_pairs`(one BIOMASS product per ALS tile, ≤ 2 years apart). Start with `N_TILES = 30`, or `None` for all tiles 

In [ ]:
from pyproj import Transformer

als_by_id = gdf_als.drop_duplicates("id").set_index("id")["item"]
fh_by_id = gdf_fh.drop_duplicates("id").set_index("id")["item"]


def compare_pair(als_item, fh_item, min_cover=0.8, return_grids=False): #change her min_cover if I want 100% of pixels or not
    """Open one ALS/BIOMASS pair, put ALS on the BIOMASS grid, return one row per BIOMASS pixel."""
    with rio.Env(GDAL_HTTP_HEADERS=f"Authorization: Bearer {TOKEN}"):
        als = rioxarray.open_rasterio(als_item.assets[ALS_ASSET].href, masked=True).squeeze().load()
        b = als.rio.transform_bounds("EPSG:4326")
        fh = (rioxarray.open_rasterio(fh_item.assets[FH_ASSET].href, masked=True).squeeze()
              .rio.clip_box(*b, crs="EPSG:4326").load())
        fhq = (rioxarray.open_rasterio(fh_item.assets[FHQ_ASSET].href, masked=True).squeeze()
               .rio.clip_box(*b, crs="EPSG:4326").load())
    if fhq.ndim > 2:
        fhq = fhq.isel(band=0)
    fhq = fhq.rio.reproject_match(fh)

    als_mean = als.rio.reproject_match(fh, resampling=Resampling.average)
    als_max = als.rio.reproject_match(fh, resampling=Resampling.max)
    valid = (als.notnull().astype("float32")
             .rio.write_crs(als.rio.crs).rio.write_nodata(np.nan))
    coverage = valid.rio.reproject_match(fh, resampling=Resampling.average)

    xx, yy = np.meshgrid(fh.x.values, fh.y.values)
    lon, lat = Transformer.from_crs(fh.rio.crs, "EPSG:4326", always_xy=True).transform(xx, yy)
    df = pd.DataFrame({
        "lon": lon.ravel(), "lat": lat.ravel(),
        "fh": fh.values.ravel(), "fh_quality": fhq.values.ravel(),
        "als_mean": als_mean.values.ravel(), "als_max": als_max.values.ravel(),
        "cover": coverage.values.ravel(),
    }).dropna(subset=["fh", "als_max", "cover"])
    df = df[df["cover"] >= min_cover]
    df["id_als"], df["id_fh"] = als_item.id, fh_item.id

    if return_grids:
        return df, {"fh": fh, "als_max": als_max, "als_mean": als_mean, "cover": coverage}
    return df

In [ ]:
N_TILES = 20          # None = all tiles in best_pairs
todo = best_pairs if N_TILES is None else best_pairs.head(N_TILES)

results, failed = [], []
for _, p in tqdm(todo.iterrows(), total=len(todo), desc="pairs"):
    try:
        d = compare_pair(als_by_id[p["id_als"]], fh_by_id[p["id_fh"]])
        d["dt_days"] = p["dt_days"]
        results.append(d)
    except Exception as e:
        failed.append((p["id_als"], str(e)[:120]))

allpix = pd.concat(results, ignore_index=True)
allpix.to_csv("als_vs_fh_pixels.csv", index=False)   

print(f"Pairs processed: {len(results)} | failed: {len(failed)} | matched pixels: {len(allpix)}")
if failed:
    print("First failures:", failed[:3])  

In [ ]:
REF = "als_mean"       # or als_max
allpix["resid"] = allpix["fh"] - allpix[REF]


def stats(y_ref, y_est):
    d = y_est - y_ref
    r = np.corrcoef(y_ref, y_est)[0, 1] if len(d) > 2 else np.nan
    return {"n": len(d), "bias": d.mean(), "RMSE": np.sqrt((d**2).mean()),
            "rel_RMSE_%": 100 * np.sqrt((d**2).mean()) / y_ref.mean(), "R2": r**2}


# Overall agreement
print(pd.Series(stats(allpix[REF], allpix["fh"])).round(2).to_string())

# One row per ALS tile
tile = (allpix.groupby("id_als")
        .apply(lambda g: pd.Series({**stats(g[REF], g["fh"]),
                                    "lon": g["lon"].mean(), "lat": g["lat"].mean(),
                                    "als_height": g[REF].mean(),
                                    "dt_days": g["dt_days"].iloc[0]}))
        .reset_index())
tile = tile[tile["n"] >= 10]          # ignore tiles with very few pixels
tile.sort_values("RMSE").round(2).head(10)

In [ ]:
### Plot scatter with all points (ALS mean height >= 1 m)
from matplotlib.colors import LogNorm

ref = "als_mean"        # x axis: ALS mean height per BIOMASS pixel
MIN_ALS_H = 1.0         # drop pixels with ALS mean height below this threshold

x, y = allpix[ref].values, allpix["fh"].values
ok = np.isfinite(x) & np.isfinite(y)
n_before = ok.sum()
ok &= x >= MIN_ALS_H
x, y = x[ok], y[ok]
print(f"Removed {n_before - ok.sum():,} pixels with ALS < {MIN_ALS_H} m "
      f"({100 * (n_before - ok.sum()) / n_before:.1f}%), kept {ok.sum():,}")

s = stats(pd.Series(x), pd.Series(y))
slope, intercept = np.polyfit(x, y, 1)          
lim = float(np.nanpercentile(np.concatenate([x, y]), 99.5))

fig, ax = plt.subplots(figsize=(8, 7.5))
hb = ax.hexbin(x, y, gridsize=80, mincnt=1, cmap="viridis", norm=LogNorm(),
               extent=(0, lim, 0, lim))
plt.colorbar(hb, ax=ax, label="Number of pixels (log scale)")

xx = np.array([0, lim])
ax.plot(xx, xx, "k--", lw=1.2, label="1:1")
ax.plot(xx, slope * xx + intercept, color="tab:red", lw=2,
        label=f"Fit: FH = {slope:.2f} · ALS + {intercept:.1f}")

ax.set_xlim(0, lim); ax.set_ylim(0, lim); ax.set_aspect("equal")
ax.set_xlabel("ALS mean canopy height [m]")
ax.set_ylabel("BIOMASS L2A forest height [m]")
ax.set_title(f"Number of ALS tiles: {allpix['id_als'].nunique()}, n = {s['n']:,} pixels "
             f"(ALS ≥ {MIN_ALS_H:g} m)\n"
             f"bias = {s['bias']:.1f} m · RMSE = {s['RMSE']:.1f} m · "
             f"rel. RMSE = {s['rel_RMSE_%']:.0f}% · R² = {s['R2']:.2f}")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

### 9. Maps: where does BIOMASS match ALS?
Red = BIOMASS higher than ALS, blue = lower, white = no bias.

In [ ]:
# 9a. One point per ALS tile, coloured by bias and by RMSE (point size = number of pixels)
blim = np.nanpercentile(np.abs(tile["bias"]), 95)
size = np.clip(tile["n"], 20, 250)

fig, ax = plt.subplots(1, 2, figsize=(17, 7), sharex=True, sharey=True)
for a in ax:
    gdf_als.boundary.plot(ax=a, color="lightgrey", lw=0.5)
    a.set_xlabel("Longitude"); a.set_ylabel("Latitude"); a.set_aspect("equal")

sc = ax[0].scatter(tile["lon"], tile["lat"], c=tile["bias"], cmap="RdBu_r",
                   vmin=-blim, vmax=blim, s=size, edgecolor="k", lw=0.3)
plt.colorbar(sc, ax=ax[0], shrink=0.7, label="Bias, BIOMASS − ALS [m]")
ax[0].set_title("Bias per ALS tile")

sc = ax[1].scatter(tile["lon"], tile["lat"], c=tile["RMSE"], cmap="YlOrRd",
                   vmin=0, vmax=np.nanpercentile(tile["RMSE"], 95), s=size, edgecolor="k", lw=0.3)
plt.colorbar(sc, ax=ax[1], shrink=0.7, label="RMSE [m]")
ax[1].set_title("RMSE per ALS tile")
plt.tight_layout()
plt.show()

In [ ]:
# 9b. Pixel residuals aggregated on a regular grid over the whole region
GRID = 40   # number of hexagons across; lower = coarser, more pixels per cell
rlim = np.nanpercentile(np.abs(allpix["resid"]), 95)

fig, ax = plt.subplots(1, 2, figsize=(17, 7), sharex=True, sharey=True)
hb = ax[0].hexbin(allpix["lon"], allpix["lat"], C=allpix["resid"], reduce_C_function=np.median,
                  gridsize=GRID, mincnt=5, cmap="RdBu_r", vmin=-rlim, vmax=rlim)
plt.colorbar(hb, ax=ax[0], shrink=0.7, label="Median BIOMASS − ALS [m]")
ax[0].set_title("Median residual")

hb = ax[1].hexbin(allpix["lon"], allpix["lat"], C=allpix["resid"],
                  reduce_C_function=lambda v: np.sqrt(np.mean(np.square(v))),
                  gridsize=GRID, mincnt=5, cmap="YlOrRd", vmin=0)
plt.colorbar(hb, ax=ax[1], shrink=0.7, label="RMSE [m]")
ax[1].set_title("RMSE")
for a in ax:
    gdf_als.boundary.plot(ax=a, color="grey", lw=0.3)
    a.set_xlabel("Longitude"); a.set_ylabel("Latitude"); a.set_aspect("equal")
plt.tight_layout()
plt.show()

In [ ]:
# 9c. Interactive map: hover a tile to see its statistics
from matplotlib import colors as mcolors
from matplotlib import cm

norm = mcolors.TwoSlopeNorm(vmin=-blim, vcenter=0, vmax=blim)
m = folium.Map(location=[tile["lat"].mean(), tile["lon"].mean()], zoom_start=6,
               tiles="Esri.WorldImagery")
for _, t in tile.iterrows():
    folium.CircleMarker(
        location=[t["lat"], t["lon"]], radius=4 + 8 * min(t["n"], 250) / 250,
        color="black", weight=0.5, fill=True, fill_opacity=0.9,
        fill_color=mcolors.to_hex(cm.RdBu_r(norm(np.clip(t["bias"], -blim, blim)))),
        tooltip=(f"{t['id_als']}<br>bias {t['bias']:.1f} m · RMSE {t['RMSE']:.1f} m · "
                 f"R² {t['R2']:.2f}<br>n = {int(t['n'])} · ALS height {t['als_height']:.1f} m · "
                 f"gap {int(t['dt_days'])} days"),
    ).add_to(m)
m

In [ ]:
# 9d. Zoom in: best and worst matching tiles, pixel by pixel
def plot_tile(id_als, title=""):
    p = best_pairs.set_index("id_als").loc[id_als]
    _, g = compare_pair(als_by_id[id_als], fh_by_id[p["id_fh"]], return_grids=True)
    ref = g[REF].where(g["cover"] >= 0.8)
    fh_t = g["fh"].where(ref.notnull())
    vmax = float(np.nanpercentile(np.concatenate([fh_t.values.ravel(), ref.values.ravel()]), 99))
    dlim = float(np.nanpercentile(np.abs((fh_t - ref).values), 95))

    fig, ax = plt.subplots(1, 3, figsize=(17, 5))
    fh_t.plot(ax=ax[0], vmin=0, vmax=vmax, cmap="viridis", cbar_kwargs={"label": "m"})
    ref.plot(ax=ax[1], vmin=0, vmax=vmax, cmap="viridis", cbar_kwargs={"label": "m"})
    (fh_t - ref).plot(ax=ax[2], cmap="RdBu_r", vmin=-dlim, vmax=dlim, cbar_kwargs={"label": "m"})
    for a, t in zip(ax, ["BIOMASS FH", f"ALS ({REF})", "BIOMASS − ALS"]):
        a.set_title(t); a.set_aspect("equal")
    fig.suptitle(f"{title}  {id_als}")
    plt.tight_layout()
    plt.show()


good = tile.sort_values("RMSE").iloc[0]
bad = tile.sort_values("RMSE").iloc[-1]
plot_tile(good["id_als"], f"Best match (RMSE {good['RMSE']:.1f} m):")
plot_tile(bad["id_als"], f"Worst match (RMSE {bad['RMSE']:.1f} m):")

### 10. What drives the differences?
Each panel splits the residuals by one possible cause.

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(15, 11))

# a) forest height: saturation in tall forest / overestimation in low forest?
hb = ax[0, 0].hexbin(allpix[REF], allpix["resid"], gridsize=50, mincnt=1, cmap="magma_r")
ax[0, 0].axhline(0, color="k", lw=1)
ax[0, 0].set_xlabel(f"ALS height ({REF}) [m]"); ax[0, 0].set_ylabel("BIOMASS − ALS [m]")
ax[0, 0].set_title("Residual vs forest height")
plt.colorbar(hb, ax=ax[0, 0], label="pixels")

# b) time between ALS and BIOMASS: forest change?
allpix["gap_class"] = pd.cut(allpix["dt_days"].abs(), bins=[0, 180, 365, 545, 730])
allpix.boxplot(column="resid", by="gap_class", ax=ax[0, 1], showfliers=False)
ax[0, 1].axhline(0, color="k", lw=1)
ax[0, 1].set_xlabel("Days between ALS and BIOMASS"); ax[0, 1].set_ylabel("BIOMASS − ALS [m]")
ax[0, 1].set_title("Residual vs time gap")

# c) BIOMASS quality layer: does it flag the bad pixels?
allpix["q_class"] = pd.qcut(allpix["fh_quality"], q=5, duplicates="drop")
allpix.boxplot(column="resid", by="q_class", ax=ax[1, 0], showfliers=False, rot=30)
ax[1, 0].axhline(0, color="k", lw=1)
ax[1, 0].set_xlabel("FH quality value (quintiles)"); ax[1, 0].set_ylabel("BIOMASS − ALS [m]")
ax[1, 0].set_title("Residual vs BIOMASS quality")

# d) per tile: is the error larger in taller forest?
ax[1, 1].scatter(tile["als_height"], tile["RMSE"], s=np.clip(tile["n"], 20, 250),
                 color="tab:blue", alpha=0.7, edgecolor="k", lw=0.3)
ax[1, 1].set_xlabel("Mean ALS height of the tile [m]"); ax[1, 1].set_ylabel("Tile RMSE [m]")
ax[1, 1].set_title("Tile RMSE vs tile height (size = pixels)")

fig.suptitle("")
plt.tight_layout()
plt.show()

# Same numbers as a table
pd.DataFrame({q: stats(g[REF], g["fh"]) for q, g in allpix.groupby("q_class", observed=True)}).T.round(2)

### 11. Compare different forest densities and sites
- **Compare ALS/CHM**: Make different groups for different forest height and see if validation improves?!?